In [1]:
using LinearAlgebra, Distributions, Random, Plots, LaTeXStrings, DataFrames, CSV, Optim, TracyWidomBeta, Measures

ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.


In [ ]:
include(joinpath(@__DIR__, "..", "..", "AuxiliaryFunctions.jl"))
include(joinpath(@__DIR__, "..", "..", "SpikeEstimation.jl"))
include(joinpath(@__DIR__, "..", "..", "BEMA.jl"))
include(joinpath(@__DIR__, "..", "..", "PA.jl"))
figdir = "Figures"
tbdir = "Tables"

DDPA (generic function with 1 method)

In [ ]:
Random.seed!(1234)

N = 8000 
d = 0.5
M = convert(Int64,ceil(N/d))
σ = 1
δvec = vcat(1.5:0.05:1.6, 1.625:0.025:2.3, 2.35:0.05:3)
σvec = ones(N)
sqrtΣ = Diagonal(sqrt.(σvec))
X = randn(Float64,N,M)
W = Matrix{Float64}(undef,N,N)

t = TWquant(0.1)
MPquants = quantMP(N,d)

vecNbr = [1,50,100,200]
lenδ = length(δvec)
SampleNbr = 50 
Percent = zeros(Float64,lenδ,6)
Avrg = zeros(Float64,lenδ,6)
Time = zeros(Float64,lenδ,6)
EvalTime = zeros(Float64,lenδ)

function process_sample(X, W, sqrtΣ, N, M, d, SpikeNbr, vecNbr, t, MPquants)
    locPercent = zeros(Float64,6)
    locAvrg = zeros(Float64,6)
    locTime = zeros(Float64,6)
    randn!(X)
    rmul!(X, inv(sqrt(M)))
    mul!(X,sqrtΣ,X)
    mul!(W,X,X')
    time_evals = @elapsed begin
        evals = eigvals(Symmetric(W))
    end
    time_BEMA0 = @elapsed begin
        BEMA0Out = BEMA0(evals,d,t,MPquants)
        locPercent[1] = BEMA0Out==SpikeNbr ? 1 : 0
        locAvrg[1] = BEMA0Out
    end
    time_DDPA = @elapsed begin
        DDPAOut = DDPA(evals,N,d)
        locPercent[2] = DDPAOut==SpikeNbr ? 1 : 0
        locAvrg[2] = DDPAOut
    end
    time_Lan1 = @elapsed begin
        results = AsympSCM(W, vecNbr=vecNbr[1], tol = 3/sqrt(N), compute_density = false)
        Nbr = results[:spikes_nbr]
        locPercent[3] = Nbr==SpikeNbr ? 1 : 0
        locAvrg[3] = Nbr
    end
    time_Lan2 = @elapsed begin
        results = AsympSCM(W, vecNbr=vecNbr[2], tol = 3/sqrt(N), compute_density = false)
        Nbr = results[:spikes_nbr]
        locPercent[4] = Nbr==SpikeNbr ? 1 : 0
        locAvrg[4] = Nbr
    end
    time_Lan3 = @elapsed begin
        results = AsympSCM(W, vecNbr=vecNbr[3], tol = 3/sqrt(N), compute_density = false)
        Nbr = results[:spikes_nbr]
        locPercent[5] = Nbr==SpikeNbr ? 1 : 0
        locAvrg[5] = Nbr
    end
    time_Lan4 = @elapsed begin
        results = AsympSCM(W, vecNbr=vecNbr[4], tol = 3/sqrt(N), compute_density = false)
        Nbr = results[:spikes_nbr]
        locPercent[6] = Nbr==SpikeNbr ? 1 : 0
        locAvrg[6] = Nbr
    end
    locTime[1] = time_evals+time_BEMA0
    locTime[2] = time_evals+time_DDPA
    locTime[3] = time_Lan1
    locTime[4] = time_Lan2
    locTime[5] = time_Lan3
    locTime[6] = time_Lan4
    return (locPercent,locAvrg,locTime,time_evals)
end

sqrtΣ0 = copy(sqrtΣ)
sqrtΣ0[1,1] = sqrt(6)
sqrtΣ0[2,2] = sqrt(5)
sqrtΣ0[3,3] = sqrt(1.5)
process_sample(X, W, sqrtΣ0, N, M, d, 2, vecNbr, t, MPquants)

for i=1:lenδ
    δ = δvec[i]
    sqrtΣ[1,1] = sqrt(6)
    sqrtΣ[2,2] = sqrt(5)
    sqrtΣ[3,3] = sqrt(δ)
    SpikeNbr = δ≤1+sqrt(d) ? 2 : 3
    results = map(_-> process_sample(X, W, sqrtΣ, N, M, d, SpikeNbr, vecNbr, t, MPquants), 1:SampleNbr)
    locPercent = first.(results)
    locAvrg = getindex.(results, 2)
    locTime = getindex.(results, 3)
    loctime_evals = getindex.(results, 4)

    Percent[i,:] = vec(reduce( .+, locPercent))/SampleNbr
    Avrg[i,:] = vec(reduce( .+, locAvrg))/SampleNbr
    Time[i,:] = vec(reduce( .+, locTime))/SampleNbr
    EvalTime[i] = sum(loctime_evals)/SampleNbr

    tb = DataFrame(A=δvec[1:i],B=Percent[1:i,1],C=Percent[1:i,2],D=Percent[1:i,3],E=Percent[1:i,4],F=Percent[1:i,5],G=Percent[1:i,6])
    CSV.write(joinpath(tbdir, "Percent.csv"),tb)
    tb = DataFrame(A=δvec[1:i],B=Avrg[1:i,1],C=Avrg[1:i,2],D=Avrg[1:i,3],E=Avrg[1:i,4],F=Avrg[1:i,5],G=Avrg[1:i,6])
    CSV.write(joinpath(tbdir, "Avrg.csv"),tb)
    tb = DataFrame(A=δvec[1:i],B=Time[1:i,1],C=Time[1:i,2],D=Time[1:i,3],E=Time[1:i,4],F=Time[1:i,5],G=Time[1:i,6])
    CSV.write(joinpath(tbdir, "Time.csv"),tb)
    tb = DataFrame(A=δvec[1:i],B=EvalTime[1:i])
    CSV.write(joinpath(tbdir, "EValTime.csv"),tb)
end

In [ ]:
labels = ["BEMA0","DDPA", "Lanczos with k = 1", "Lanczos with k = 50", "Lanczos with k = 100", "Lanczos with k = 200"]
markers = [:square, :diamond, :circle, :circle, :circle, :circle]
colors  = [:blue, :orange, :gray, :purple, :green, :red]

figsize = (1600, 400)

guidef = font(16)
tickf  = font(14)
legendf = font(14)

leftm = 15mm
bottomm = 15mm
topm = 5mm
rightm = 5mm

p1 = plot(xlabel = L"\delta", ylabel = "Probability of correct estimation", legend = :outerright, ylim = (-0.02, 1.02), grid = true, size = figsize, framestyle=:box, guidefont = guidef, tickfont = tickf, legendfont = legendf, left_margin = leftm, bottom_margin = bottomm, top_margin = topm, right_margin = rightm)

for i in 1:6
    plot!(p1, δvec, Percent[:, i], label = labels[i], marker = markers[i], markersize = 6, linewidth = 2, color = colors[i])
end

p2 = plot(xlabel = L"\delta", ylabel = "Time (in seconds)", legend = :outerright, grid = true, size = figsize, framestyle=:box, guidefont = guidef, tickfont = tickf, legendfont = legendf, left_margin = leftm, bottom_margin = bottomm, top_margin = topm, right_margin = rightm)

for i in 1:6
    plot!(p2, δvec, Time[:, i], label = labels[i], marker = markers[i], markersize = 6, linewidth = 2, color = colors[i])
end

savefig(p1,joinpath(figdir, "Prob.pdf"))
savefig(p2,joinpath(figdir, "Time.pdf"))


"/Users/user/Library/CloudStorage/OneDrive-UW/Research/UW/FastSpikeDetection/Sim2Time.pdf"